# Hope Speech Detection — XLM-R Fine-tuning (Step 4)

Fine-tunes `xlm-roberta-base` for binary hope speech classification
(0 = Non-hope, 1 = Hope) on the preprocessed HopeEDI data.

**How to use**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Set `LANG` in the Configuration cell (`english`, `tamil`, or `malayalam`)
3. Runtime → Run all. Rerun the notebook once per language.

**Inputs:** `MyDrive/hope_speech/processed/{lang}_{split}_processed.csv`
**Outputs (saved to Drive):**
- model: `MyDrive/hope_speech/xlmr_models/{lang}/`
- metrics: `MyDrive/hope_speech/metrics/xlmr_{lang}_metrics.json`

Handles class imbalance with a class-weighted loss (same philosophy as the
baseline's `class_weight="balanced"`), picks the best epoch by **dev macro F1**,
and evaluates once on test at the end.

In [ ]:
# ============================================================
# 0. Environment
# ============================================================
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate evaluate

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# ============================================================
# 1. Configuration  (EDIT LANG, rerun notebook per language)
# ============================================================
LANG = "english"          # "english" | "tamil" | "malayalam"

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128             # covers ~99% of comments
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/hope_speech")
DATA_DIR = BASE_DIR / "processed"          # the 9 processed CSVs live here
MODEL_OUT = BASE_DIR / "xlmr_models" / LANG
METRICS_OUT = BASE_DIR / "metrics"
CKPT_DIR = Path(f"/content/checkpoints_{LANG}")   # local scratch (fast)
MODEL_OUT.mkdir(parents=True, exist_ok=True)
METRICS_OUT.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / f"{LANG}_train_processed.csv").exists(), \
    f"Data not found in {DATA_DIR} — check the Drive upload"
print("Config OK — training", LANG)

In [ ]:
# ============================================================
# 2. Load data
# ============================================================
import pandas as pd

def load_split(split):
    df = pd.read_csv(DATA_DIR / f"{LANG}_{split}_processed.csv")
    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.strip() != ""]
    assert set(df["label"].unique()) <= {0, 1}
    return df[["text", "label"]].reset_index(drop=True)

train_df, dev_df, test_df = load_split("train"), load_split("dev"), load_split("test")
print(f"train: {len(train_df)}  (Hope: {train_df.label.sum()})")
print(f"dev:   {len(dev_df)}   (Hope: {dev_df.label.sum()})")
print(f"test:  {len(test_df)}   (Hope: {test_df.label.sum()})")

In [ ]:
# ============================================================
# 3. Tokenize
# ============================================================
from datasets import Dataset
from transformers import AutoTokenizer, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds = {}
for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    d = Dataset.from_pandas(df, preserve_index=False)
    ds[name] = d.map(tokenize, batched=True, remove_columns=["text"])
print(ds["train"])

In [ ]:
# ============================================================
# 4. Model + class-weighted Trainer
# ============================================================
import numpy as np
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding,
                          EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    "balanced", classes=np.array([0, 1]), y=train_df["label"].values)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).cuda()
print("class weights:", class_weights)   # upweights the Hope class

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "Non_hope_speech", 1: "Hope_speech"},
    label2id={"Non_hope_speech": 0, "Hope_speech": 1})

class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy (imbalance handling)."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights_t)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "hope_f1": f1_score(labels, preds, pos_label=1),
        "hope_precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "hope_recall": recall_score(labels, preds, pos_label=1, zero_division=0),
    }

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["dev"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# ============================================================
# 5. Train  (~20-45 min on T4 depending on language)
# ============================================================
trainer.train()
print("\nBest dev macro F1:", trainer.state.best_metric)

In [ ]:
# ============================================================
# 6. Test evaluation + save everything to Drive
# ============================================================
import json
from sklearn.metrics import classification_report, confusion_matrix

test_out = trainer.predict(ds["test"])
test_metrics = {k.replace("test_", ""): round(v, 4)
                for k, v in test_out.metrics.items()
                if k.startswith("test_") and isinstance(v, float)}
preds = np.argmax(test_out.predictions, axis=-1)
labels = test_df["label"].values
test_metrics["confusion_matrix"] = confusion_matrix(labels, preds).tolist()

print(classification_report(labels, preds, digits=3,
                            target_names=["Non-hope (0)", "Hope (1)"]))
print("Confusion matrix [[TN FP] [FN TP]]:")
print(confusion_matrix(labels, preds))

dev_out = trainer.evaluate(ds["dev"])
payload = {
    "language": LANG,
    "model": MODEL_NAME,
    "hyperparams": {"max_len": MAX_LEN, "batch_size": BATCH_SIZE,
                    "epochs": EPOCHS, "lr": LEARNING_RATE, "seed": SEED},
    "dev": {k.replace("eval_", ""): round(v, 4) for k, v in dev_out.items()
            if isinstance(v, float)},
    "test": test_metrics,
}
metrics_path = METRICS_OUT / f"xlmr_{LANG}_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(payload, f, indent=2)
print("\nMetrics saved ->", metrics_path)

trainer.save_model(str(MODEL_OUT))
tokenizer.save_pretrained(str(MODEL_OUT))
print("Model saved   ->", MODEL_OUT)

In [ ]:
# ============================================================
# 7. Sanity check: live predictions (same spirit as predict.py)
# ============================================================
SAMPLES = {
    "english": ["there is hope", "no hope left", "never lose hope",
                "Just smile and stop being weak"],
    "tamil": ["எனக்கு நம்பிக்கை இருக்கிறது", "நம்பிக்கை இல்லை",
              "இது நல்ல நாள் வரும்", "நம்மால் முடியாது"],
    "malayalam": ["ആശ ഇല്ല", "ഞങ്ങൾ വിജയിക്കും",
                  "നല്ല ദിവസം വരും", "ഇത് സാധ്യമല്ല"],
}
texts = SAMPLES[LANG]
enc = tokenizer(texts, truncation=True, max_length=MAX_LEN,
                padding=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model(**enc)
preds = out.logits.argmax(-1).cpu().tolist()
for t, p in zip(texts, preds):
    print(f"{p}  ({'Hope' if p==1 else 'Non-hope'})  {t}")